# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset package using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library, referencing all entities by their `@id` fields per the Croissant schema. You will learn how to:

- Load Croissant metadata from a remote FAIR² dataset
- Explore available record sets and fields using their `@id`s
- Load data for these record sets into pandas DataFrames
- Perform exploratory data analysis (EDA) and data transformations by referencing fields by `@id`
- Visualize results with plots

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed (uncomment if running on Colab/local)
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display dataset summary
print(f"{getattr(metadata, 'name', '<No name>')}\n\n{getattr(metadata, 'description', '<No description>')}")

## 2. Data Overview
Review available record sets, fields, their `@id`s, and key properties. This is essential for referencing and extracting data precisely using the Croissant schema.

Below, we list all record sets, their `@id`s, and their fields' `@id`s.

In [ ]:
# List all record sets with their @id and field @ids.
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets found in dataset (record_set/@id array is empty).\nPlease check dataset schema or contact the owner if you expect tabular data.")
else:
    for rs in record_sets:
        print(f"Record set @id: {rs['@id']}")
        # Fields are referenced by 'field' property
        fields = rs.get('field', []) if isinstance(rs.get('field', []), list) else [rs.get('field', [])]
        print(" Fields:")
        for f in fields:
            if hasattr(f, 'get'):
                print(f"  - {f.get('@id')}")
            else:
                print(f"  - {f}")
        print()
# For demonstration, print the list of record sets' @id fields.
record_set_ids = [rs['@id'] for rs in record_sets] if record_sets else []
print(f"Record set @ids: {record_set_ids if record_set_ids else '[None]'}")

## 3. Data Extraction
Load data from each available record set into pandas DataFrames for further analysis.

All data operations reference entities by their `@id` fields as per the Croissant schema.

In [ ]:
# Extract all available record sets as DataFrames.
dataframes = {}
if record_set_ids:
    for record_set_id in record_set_ids:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded record set: {record_set_id}, shape: {dataframes[record_set_id].shape}")
    # Preview the first available record set (if any)
    sample_id = record_set_ids[0]
    print(f"\nColumns in record set '{sample_id}': {list(dataframes[sample_id].columns)}\n")
    display(dataframes[sample_id].head())
else:
    print("No record sets are defined, so no tabular data was extracted.")

## 4. Exploratory Data Analysis (EDA)
Perform exploratory data analysis and basic transformations, referencing fields by their `@id` fields only. We'll:
- Select a numeric field for filtering and normalization
- Filter rows based on a threshold
- Normalize the numeric field
- Optionally group data by another field (referenced by its `@id`)

_**Note**: If the record sets or field `@id`s discovered above are empty, update the code below as needed using correct `@id` values from your overview output._

In [ ]:
# Only run EDA if a record set was loaded.
import numpy as np

if record_set_ids and dataframes[record_set_ids[0]].shape[1] > 0:
    # Example: Use the first record set and identify numeric field by @id.
    example_rs_id = record_set_ids[0]
    df = dataframes[example_rs_id]
    print(f"Available columns (@id) for EDA in '{example_rs_id}':\n{list(df.columns)}\n")
    
    # Pick a numeric field by its @id (Replace as appropriate!)
    # For demonstration, select the first column with numeric type
    numeric_field_id = None
    for col in df.columns:
        try:
            # Try to coerce to numeric
            sample = pd.to_numeric(df[col], errors='coerce')
            if sample.notnull().sum() > 0:
                numeric_field_id = col
                break
        except Exception:
            continue
    if numeric_field_id is not None:
        print(f"Using numeric field '@id': {numeric_field_id}")
        # Convert to numeric for EDA
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].mean() # Example: use mean as threshold
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"\nFiltered records from '{example_rs_id}' with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        field_norm = f"{numeric_field_id}_normalized"
        filtered_df[field_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, field_norm]].head())

        # Try grouping by another field (categorical, by @id)
        group_field = None
        for col in df.columns:
            if col != numeric_field_id and df[col].nunique() < min(8, df.shape[0]//5):  # Heuristic: few unique values ~category
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field, as_index=False)[numeric_field_id].mean()
            print(f"\nGrouped means of {numeric_field_id} by '{group_field}':")
            display(grouped_df.head())
        else:
            print("\nNo suitable categorical (@id) field found for grouping.")
    else:
        print("No numeric fields detected for EDA in this record set.")
else:
    print("No loaded data is available for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field, or the relationship between fields (by their `@id`s) in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'filtered_df' in locals() and numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field_id].dropna(), kde=True, bins=10, color='royalblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No filtered data available for visualization.")

## 6. Conclusion
This notebook demonstrated:
- How to load a FAIR² Croissant dataset by schema URL with the `mlcroissant` library
- Exploration of available record sets, fields, and their `@id`s, using schema-driven references
- Extraction of tabular data for each record set
- Basic EDA and visualization, strictly referencing all entities by their `@id`

You can further adapt this notebook with domain-specific analyses, new statistical summaries, and more complex visualizations as required.